HW2 Part 1: Data cleaning

I use the reference table ohlc (same stocks and dates) for three things: the trading calendar, the adjustment factor of each day, and the check of the large returns. ohlc_hw itself is never modified.

In [1]:
import os
import sqlite3

import numpy as np
import pandas as pd
from IPython.display import display

DB_PATH = "data/data.db"   # copy of the course database, never committed

PRICES = ["open", "high", "low", "close"]
VALUES = PRICES + ["adj_close", "volume"]
KEY = ["ticker", "ts"]                             # one row = one stock on one day

1.1

In [2]:
if not os.path.exists(DB_PATH):
    raise FileNotFoundError("Put a copy of the course database (bCourses) in data/data.db, then run again.")
conn = sqlite3.connect(DB_PATH)
raw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn)
ref = pd.read_sql_query("SELECT ticker, ts, close, adj_close FROM ohlc", conn)
print("ohlc_hw (homework data):", raw.shape, "| ohlc (reference):", ref.shape)
raw.info()

ohlc_hw (homework data): (397711, 8) | ohlc (reference): (368251, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


In [3]:
display(raw.describe())

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


Columns: ts is the date, ticker the stock, open high low close the raw prices of the day, adj_close the close adjusted for splits and dividends, volume the number of shares traded.

describe() already shows two problems: open has a minimum of 0, and adj_close goes up to 10,442 while close is never above 606.

In [4]:
print("Date range:", raw["ts"].min(), "to", raw["ts"].max())
print("Different ticker names:", raw["ticker"].nunique())
print(raw["ticker"].drop_duplicates().head(6).tolist())

Date range: 1962-01-02 to 2024-12-06
Different ticker names: 150
['CAT', 'CVX', 'HD', 'AAPL', 'TRV', 'JNJ']


1.2

I clean the ticker names first (HTML tags, spaces, capitals), because the same stock appears under several names.

In [5]:
def clean_ticker(labels):
    """Remove the HTML tags and the spaces, and write the names in capital letters."""
    return labels.astype(str).str.replace(r"<[^>]*>", "", regex=True).str.strip().str.upper()


df = raw.copy()
df["raw_ticker"] = df["ticker"]
df["ticker"] = clean_ticker(df["ticker"])
df["ts"] = pd.to_datetime(df["ts"])

ref["ticker"] = clean_ticker(ref["ticker"])
ref["ts"] = pd.to_datetime(ref["ts"])
ref["factor"] = ref["adj_close"] / ref["close"]     # adjustment factor of each day

print(raw["ticker"].nunique(), "names ->", df["ticker"].nunique(), "tickers:")
print(", ".join(sorted(df["ticker"].unique())))

150 names -> 30 tickers:
AAPL, AMGN, AMZN, AXP, BA, CAT, CRM, CSCO, CVX, DIS, GS, HD, HON, IBM, JNJ, JPM, KO, MCD, MMM, MRK, MSFT, NKE, NVDA, PG, SHW, TRV, UNH, V, VZ, WMT


In [6]:
exact_dup = df.duplicated(KEY + VALUES)
unique_rows = df[~exact_dup]
conflict = unique_rows.duplicated(KEY, keep=False)
print("Exact duplicates:", exact_dup.sum())
print("Rows in conflicting duplicates:", conflict.sum())

Exact duplicates: 29460
Rows in conflicting duplicates: 0


In [7]:
check = pd.DataFrame({"missing": unique_rows[VALUES].isna().sum(),
                      "infinite": np.isinf(unique_rows[VALUES]).sum()})
display(check)

,missing,infinite
open,0,0
high,0,0
low,0,0
close,0,0
adj_close,18374,0
volume,0,0


Calendar for the missing days: the dates of each stock in ohlc, between its first and last date, because the weekdays alone would count the holidays as missing. I leave the weekends out since the NYSE is closed, and ohlc has some Sunday rows.

In [8]:
def find_missing_days(data, calendar):
    """For each stock, the days of the calendar (between its first and last date) with no row in data."""
    rows = []
    for ticker, group in data.groupby("ticker"):
        first, last = group["ts"].min(), group["ts"].max()
        expected = calendar.loc[(calendar["ticker"] == ticker) & calendar["ts"].between(first, last), "ts"]
        for day in sorted(set(expected) - set(group["ts"])):
            rows.append({"ticker": ticker, "ts": day})
    return pd.DataFrame(rows, columns=KEY)


# 5 trading days (2024-01-01 is a holiday), and 2024-01-03 removed
demo_calendar = pd.DataFrame({"ticker": "DEMO", "ts": pd.to_datetime(
    ["2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-08"])})
demo_rows = demo_calendar.drop(index=1)
demo_missing = find_missing_days(demo_rows, demo_calendar)
print(demo_missing)
assert demo_missing["ts"].tolist() == [pd.Timestamp("2024-01-03")]

  ticker         ts
0   DEMO 2024-01-03


In [9]:
calendar = ref.loc[ref["ts"].dt.dayofweek < 5, KEY]       # dates of ohlc, Monday to Friday
missing_days = find_missing_days(unique_rows, calendar)
for ticker, group in unique_rows.groupby("ticker"):
    n = (missing_days["ticker"] == ticker).sum()
    print(f"{ticker} is missing {n} trading days between {group['ts'].min().date()} and {group['ts'].max().date()}")

AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
BA is missing 0 trading days between 1962-01-02 and 2024-10-18
CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
GS is missing 0 trading days between 1999-05-04 and 2024-10-18
HD is missing 0 trading days between 1981-09-22 and 2024-10-18
HON is missing 0 trading days between 1962-01-02 and 2024-10-18
IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
JNJ is missing 0 trading days between 1962-01-02 and 2024-10-18
JPM is missing 0 trading days between 1

In [10]:
weekend = unique_rows[unique_rows["ts"].dt.dayofweek >= 5]
print("Rows with a weekend date:", len(weekend), "| days:", weekend["ts"].dt.day_name().unique(),
      "| years:", weekend["ts"].dt.year.unique(), "| tickers:", weekend["ticker"].nunique())
print("Also in the reference table:", (ref["ts"].dt.dayofweek >= 5).sum(), "rows")

Rows with a weekend date: 384 | days: ['Sunday'] | years: [1974 1975] | tickers: 16
Also in the reference table: 384 rows


No missing day. But 384 rows are dated on a Sunday (1974 and 1975, 16 stocks), also in ohlc: I exclude them in 1.3. The other days of these weeks are probably one day early too (there are no Friday rows), but I cannot prove it, so I keep them as they are.

In [11]:
def flag_extreme_returns(data, n_std=3):
    """Daily return of the close in each stock, and a flag when |return - mean| > n_std * std (data sorted by stock and date)."""
    close = data["close"].where(np.isfinite(data["close"]) & (data["close"] > 0))
    ret = close.groupby(data["ticker"]).pct_change(fill_method=None)
    mean = ret.groupby(data["ticker"]).transform("mean")
    std = ret.groupby(data["ticker"]).transform("std")      # sample std (ddof=1)
    flag = ((ret - mean).abs() > n_std * std) & (std > 0)    # a missing return gives False
    return ret, flag

Severity: high when the value is impossible and would make the prices or returns wrong, medium when the row is usable but needs a repair or a check, low when nothing is lost and it is only a flag (exact duplicate, volume of 0).

In [12]:
def detect_issues(data, ref):
    """Data Quality Report of data: one row per problem found. ref gives the adjustment factors."""
    data = data.sort_values(KEY).reset_index(drop=True)
    factor = data[KEY].merge(ref[KEY + ["factor"]], on=KEY, how="left")["factor"]
    ret, extreme = flag_extreme_returns(data)
    prices = data[PRICES + ["adj_close"]]
    checks = [
        ("missing_price", data[PRICES].isna().any(axis=1), "a raw price is missing", "high"),
        ("invalid_price", (prices <= 0).any(axis=1) | np.isinf(prices).any(axis=1),
         "a price is 0, negative or infinite", "high"),
        ("close_outside_range", (data["high"] < data["close"]) | (data["low"] > data["close"]),
         "high < close or low > close", "high"),
        ("open_outside_range", (data["open"] > data["high"]) | (data["open"] < data["low"]),
         "open outside [low, high]", "high"),
        ("high_below_low", data["high"] < data["low"], "high < low", "high"),
        ("weekend_date", data["ts"].dt.dayofweek >= 5, "weekend date (the NYSE is closed)", "high"),
        ("invalid_volume", (data["volume"] < 0) | np.isinf(data["volume"]), "volume negative or infinite", "high"),
        ("missing_adj_close", data["adj_close"].isna(), "adjusted close is missing", "medium"),
        ("adj_close_inconsistent", ((data["adj_close"] / data["close"]) / factor - 1).abs() > 1e-4,
         "adj_close / close is different from the factor of the reference", "medium"),
        ("missing_volume", data["volume"].isna(), "volume is missing", "medium"),
        ("extreme_return", extreme, "return " + (ret * 100).round(2).astype(str) + "%, more than 3 std from the mean",
         "medium"),
        ("zero_volume", data["volume"] == 0, "volume is 0 (maybe an old price repeated)", "low"),
    ]
    parts = []
    for issue_type, mask, description, severity in checks:
        part = data.loc[mask, ["ticker", "ts"]].rename(columns={"ts": "issue_date"})
        part["issue_type"] = issue_type
        part["description"] = description[mask] if isinstance(description, pd.Series) else description
        part["severity"] = severity
        parts.append(part)
    return pd.concat(parts, ignore_index=True)[["ticker", "issue_type", "issue_date", "description", "severity"]]


report = detect_issues(unique_rows[~conflict], ref)
extra = [
    df.loc[exact_dup, ["ticker", "ts"]].assign(issue_type="exact_duplicate", description="identical to another row",
                                               severity="low"),
    unique_rows.loc[conflict, ["ticker", "ts"]].assign(issue_type="conflicting_duplicate",
                                                       description="same ticker and date, different values",
                                                       severity="high"),
    missing_days.assign(issue_type="missing_trading_day", description="trading day of the reference with no row",
                        severity="medium"),
]
extra = pd.concat([part for part in extra if len(part) > 0], ignore_index=True).rename(columns={"ts": "issue_date"})
quality_report = pd.concat([report, extra], ignore_index=True).sort_values(["ticker", "issue_date"])
quality_report_before = quality_report.copy()        # kept to compare after the cleaning

display(quality_report.groupby(["issue_type", "severity"]).size().rename("issues").sort_values(ascending=False))
display(quality_report[quality_report["severity"] == "high"].head(10))

issue_type              severity
exact_duplicate         low         29460
missing_adj_close       medium      18374
adj_close_inconsistent  medium      10417
extreme_return          medium       5056
weekend_date            high          384
zero_volume             low            75
invalid_price           high            5
open_outside_range      high            5
Name: issues, dtype: int64

,ticker,issue_type,issue_date,description,severity
10,AXP,weekend_date,1974-01-06,weekend date (the NYSE is closed),high
11,AXP,weekend_date,1974-01-13,weekend date (the NYSE is closed),high
12,AXP,weekend_date,1974-01-20,weekend date (the NYSE is closed),high
13,AXP,weekend_date,1974-01-27,weekend date (the NYSE is closed),high
14,AXP,weekend_date,1974-02-03,weekend date (the NYSE is closed),high
15,AXP,weekend_date,1974-02-10,weekend date (the NYSE is closed),high
16,AXP,weekend_date,1974-02-24,weekend date (the NYSE is closed),high
17,AXP,weekend_date,1974-03-03,weekend date (the NYSE is closed),high
18,AXP,weekend_date,1974-03-10,weekend date (the NYSE is closed),high
19,AXP,weekend_date,1974-03-17,weekend date (the NYSE is closed),high


Main findings: 29,460 exact duplicates, 384 Sunday rows, 5 rows with open = 0 (first day of a stock), and 18,374 missing plus 10,417 wrong adjusted closes (4 to 24 times too high on single days, while the close is correct).

1.3

Conflicting duplicates and weekend dates: I exclude the rows, I cannot know the correct values.

Impossible prices (0, negative, infinite, or outside [low, high]): Option 1, the value becomes missing and is forward filled in the stock. If high < low, I only replace high and low, so a correct close is not lost. After the fill I check the range again: a filled close outside it becomes (high + low) / 2.

First day of a stock with no earlier price (the 5 open = 0): excluded, I prefer not to invent a price.

Adjusted close: close * factor of ohlc on the same day. I never forward fill the factor, because it changes at each split and dividend.

Large returns: an error only with a proof, a close more than 1% away from ohlc or a volume of 0 (no trade, so the price cannot move). Errors are interpolated, adjusted close included. The other flags are kept.

The excluded rows are kept apart with the reason.

In [13]:
def clean_data(raw_table, ref):
    """Clean a table like ohlc_hw with the decisions above. Returns (clean table, excluded rows, counts)."""
    data = raw_table.copy()
    data["raw_ticker"] = data["ticker"]
    data["ticker"] = clean_ticker(data["ticker"])
    data["ts"] = pd.to_datetime(data["ts"])
    counts = {"raw rows": len(data)}
    excluded = []

    # 1. Duplicates
    dup = data.duplicated(KEY + VALUES)
    counts["removed: exact duplicates"] = dup.sum()
    data = data[~dup]
    conflict = data.duplicated(KEY, keep=False)
    excluded.append(data[conflict].assign(reason="conflicting duplicate"))
    data = data[~conflict]

    # 2. Weekend dates
    weekend = data["ts"].dt.dayofweek >= 5
    excluded.append(data[weekend].assign(reason="weekend date"))
    data = data[~weekend]

    # 3. The 4 raw prices missing (before any filling)
    all_missing = data[PRICES].isna().all(axis=1)
    counts["removed: all raw prices missing"] = all_missing.sum()
    data = data[~all_missing].sort_values(KEY).reset_index(drop=True)
    for col in VALUES:
        data[col + "_repaired"] = False

    # 4. Price 0, negative or infinite -> missing
    for col in PRICES + ["adj_close"]:
        bad = (data[col] <= 0) | np.isinf(data[col])
        data.loc[bad, col] = np.nan

    # 5. Option 1: a value outside [low, high] -> missing (if high < low, only high and low)
    bad_range = data["high"] < data["low"]
    data.loc[bad_range, ["high", "low"]] = np.nan
    data.loc[(data["close"] > data["high"]) | (data["close"] < data["low"]), "close"] = np.nan
    data.loc[(data["open"] > data["high"]) | (data["open"] < data["low"]), "open"] = np.nan

    # 6. Forward fill in each stock, exclude the rows with no earlier price, then check the range of the day
    for col in PRICES:
        data[col + "_repaired"] = data[col].isna()
        data[col] = data.groupby("ticker")[col].ffill()
    no_earlier = data[PRICES].isna().any(axis=1)
    excluded.append(data[no_earlier].assign(reason="no earlier price to repair from"))
    data = data[~no_earlier].reset_index(drop=True)
    before = data[PRICES].copy()
    middle = (data["high"] + data["low"]) / 2
    fix_close = data["close_repaired"] & ((data["close"] > data["high"]) | (data["close"] < data["low"]))
    data.loc[fix_close, "close"] = middle[fix_close]
    fix_open = data["open_repaired"]
    data.loc[fix_open, "open"] = data.loc[fix_open, "open"].clip(data.loc[fix_open, "low"], data.loc[fix_open, "high"])
    data["high"] = data[["high", "open", "close"]].max(axis=1)
    data["low"] = data[["low", "open", "close"]].min(axis=1)
    for col in PRICES:
        data[col + "_repaired"] |= data[col] != before[col]          # each changed value gets its flag

    # 7. Adjusted close = close * factor of ohlc on the same day, where it is missing or wrong
    data = data.merge(ref[KEY + ["factor", "close"]].rename(columns={"close": "ref_close"}), on=KEY, how="left")
    expected = data["close"] * data["factor"]
    wrong = data["adj_close"].isna() | ((data["adj_close"] / expected - 1).abs() > 1e-4)
    data.loc[wrong, "adj_close"] = expected[wrong]
    data["adj_close_repaired"] = wrong
    no_factor = data["adj_close"].isna()
    excluded.append(data[no_factor].assign(reason="no factor in the reference for the adjusted close"))
    data = data[~no_factor].reset_index(drop=True)

    # 8. Volume
    bad_volume = data["volume"].isna() | (data["volume"] < 0) | np.isinf(data["volume"])
    data["zero_volume_flag"] = data["volume"] == 0
    data.loc[bad_volume, "volume"] = 0
    data["volume_repaired"] = bad_volume

    # 9. Large returns: error if the close is more than 1% away from ohlc, or if the volume is 0
    ret, extreme = flag_extreme_returns(data)
    off_ref = (data["close"] / data["ref_close"] - 1).abs() > 0.01
    same_stock_next = data["ticker"].eq(data["ticker"].shift(-1))
    error = (off_ref & (extreme | (extreme.shift(-1, fill_value=False) & same_stock_next))) \
        | (extreme & data["zero_volume_flag"])
    first_or_last = data["ticker"].ne(data["ticker"].shift(1)) | data["ticker"].ne(data["ticker"].shift(-1))
    at_edge = error & first_or_last                     # no day before or after: cannot interpolate
    excluded.append(data[at_edge].assign(reason="price error with no day before or after"))
    data = data[~at_edge].reset_index(drop=True)
    error = error[~at_edge].reset_index(drop=True)
    cols = PRICES + ["adj_close"]                       # adj_close too: the factor of ohlc can be wrong that day
    data.loc[error, cols] = np.nan
    data[cols] = data.groupby("ticker")[cols].transform(lambda s: s.interpolate(limit_area="inside"))
    data.loc[error, [c + "_repaired" for c in cols]] = True
    counts["price errors interpolated"] = error.sum()
    ret, data["extreme_return_flag"] = flag_extreme_returns(data)     # flags on the final prices

    excluded = pd.concat(excluded, ignore_index=True)
    counts["excluded (kept apart, with a reason)"] = len(excluded)
    counts["kept in the cleaned table"] = len(data)
    assert (counts["removed: exact duplicates"] + counts["removed: all raw prices missing"]
            + counts["excluded (kept apart, with a reason)"] + counts["kept in the cleaned table"]) == counts["raw rows"]
    return data.drop(columns=["factor", "ref_close"]), excluded, pd.Series(counts)

Some cases are not in the real data, so I test them on a small fake table with one error of each type.

In [14]:
days = pd.bdate_range("2024-01-02", periods=60)
rng = np.random.default_rng(0)
demo_parts = []
for ticker, start in [("DEMO_A", 100.0), ("DEMO_B", 50.0)]:
    close = start * np.exp(np.cumsum(rng.normal(0, 0.01, len(days))))
    demo_parts.append(pd.DataFrame({"ts": days.strftime("%Y-%m-%d"), "ticker": ticker, "open": close,
                                    "high": close * 1.01, "low": close * 0.99, "close": close,
                                    "adj_close": close * 0.98, "volume": 1_000_000.0}))
demo_ref = pd.concat(demo_parts, ignore_index=True)
demo_raw = demo_ref.copy()


def row(ticker, day):
    """Index of the demo row for this stock and this day."""
    return demo_raw.index[(demo_raw["ticker"] == ticker) & (demo_raw["ts"] == day)][0]


demo_raw.loc[row("DEMO_A", "2024-01-10"), "close"] *= 100                 # close above high
demo_raw.loc[row("DEMO_A", "2024-01-11"), "close"] = np.nan               # missing close
demo_raw.loc[row("DEMO_A", "2024-01-12"), PRICES] = np.nan                # the 4 prices missing
demo_raw.loc[row("DEMO_A", "2024-01-02"), "open"] = 0                     # open = 0 on the first day
demo_raw.loc[row("DEMO_B", "2024-01-16"), "volume"] = np.nan              # missing volume
demo_raw.loc[row("DEMO_B", "2024-01-17"), "volume"] = -10                 # negative volume
demo_raw.loc[row("DEMO_B", "2024-01-18"), "adj_close"] = np.nan           # missing adjusted close
demo_raw.loc[row("DEMO_B", "2024-02-20"), ["open", "high", "low", "close", "adj_close"]] *= 1.5   # wrong day
demo_raw.loc[row("DEMO_A", "2024-01-15"), "close"] *= 0.5                 # close below low
demo_raw.loc[row("DEMO_A", "2024-01-16"), "low"] *= 3                     # low above high
demo_raw.loc[row("DEMO_A", "2024-01-17"), "high"] = np.inf                # infinite price
demo_raw.loc[row("DEMO_B", "2024-01-19"), "adj_close"] = -5               # negative adjusted close
spike = row("DEMO_A", "2024-02-06")                                       # jump with volume 0, also in the reference
true_close = demo_ref.loc[spike, "close"]
for table in [demo_raw, demo_ref]:
    table.loc[spike, ["open", "high", "low", "close", "adj_close"]] *= 2
    table.loc[spike, "volume"] = 0
duplicate = demo_raw.loc[[row("DEMO_B", "2024-01-22")]].assign(ticker="<b>demo_b</b>")           # exact duplicate
conflicting = demo_raw.loc[[row("DEMO_B", "2024-01-24")]].assign(close=lambda d: d["close"] * 1.05)  # conflict
saturday = demo_raw.loc[[row("DEMO_B", "2024-01-26")]].assign(ts="2024-01-27")                    # weekend
demo_raw = pd.concat([demo_raw, duplicate, conflicting, saturday], ignore_index=True)

demo_ref["ticker"] = clean_ticker(demo_ref["ticker"])
demo_ref["ts"] = pd.to_datetime(demo_ref["ts"])
demo_ref["factor"] = demo_ref["adj_close"] / demo_ref["close"]

demo_std = demo_raw.assign(ticker=clean_ticker(demo_raw["ticker"]), ts=pd.to_datetime(demo_raw["ts"]))
demo_issues = detect_issues(demo_std.drop_duplicates(), demo_ref)
print("Issues found in the demo table:")
display(demo_issues["issue_type"].value_counts())
expected = {"missing_price", "invalid_price", "close_outside_range", "open_outside_range", "high_below_low",
            "weekend_date", "invalid_volume", "missing_adj_close", "adj_close_inconsistent", "missing_volume",
            "extreme_return", "zero_volume"}
assert expected <= set(demo_issues["issue_type"]), expected - set(demo_issues["issue_type"])   # each check finds its case

demo_clean, demo_excluded, demo_counts = clean_data(demo_raw, demo_ref)
display(demo_counts.to_frame("rows"))
display(demo_excluded[["ticker", "ts", "reason"]])

Issues found in the demo table:


issue_type
close_outside_range       4
adj_close_inconsistent    4
extreme_return            3
invalid_price             3
open_outside_range        2
missing_price             2
high_below_low            1
weekend_date              1
missing_adj_close         1
invalid_volume            1
missing_volume            1
zero_volume               1
Name: count, dtype: int64

,rows
raw rows,123
removed: exact duplicates,1
removed: all raw prices missing,1
price errors interpolated,2
"excluded (kept apart, with a reason)",4
kept in the cleaned table,117


,ticker,ts,reason
0,DEMO_B,2024-01-24,conflicting duplicate
1,DEMO_B,2024-01-24,conflicting duplicate
2,DEMO_B,2024-01-27,weekend date
3,DEMO_A,2024-01-02,no earlier price to repair from


In [15]:
d = demo_clean.set_index(KEY)
day = pd.Timestamp
truth = demo_ref.set_index(KEY)
assert d.loc[("DEMO_A", day("2024-01-10")), "close_repaired"]
assert d.loc[("DEMO_A", day("2024-01-10")), "low"] <= d.loc[("DEMO_A", day("2024-01-10")), "close"] \
    <= d.loc[("DEMO_A", day("2024-01-10")), "high"]
assert d.loc[("DEMO_A", day("2024-01-11")), "close_repaired"]
assert ("DEMO_A", day("2024-01-12")) not in d.index
assert ("DEMO_A", day("2024-01-02")) not in d.index
assert d.loc[("DEMO_B", day("2024-01-16")), "volume"] == 0 and d.loc[("DEMO_B", day("2024-01-16")), "volume_repaired"]
assert d.loc[("DEMO_B", day("2024-01-17")), "volume"] == 0 and d.loc[("DEMO_B", day("2024-01-17")), "volume_repaired"]
assert np.isclose(d.loc[("DEMO_B", day("2024-01-18")), "adj_close"], truth.loc[("DEMO_B", day("2024-01-18")), "adj_close"])
assert d.loc[("DEMO_B", day("2024-02-20")), "close_repaired"]
assert abs(d.loc[("DEMO_B", day("2024-02-20")), "close"] / truth.loc[("DEMO_B", day("2024-02-20")), "close"] - 1) < 0.03
assert ("DEMO_B", day("2024-01-24")) not in d.index
assert ("DEMO_B", day("2024-01-27")) not in d.index
assert not d.index.duplicated().any()
assert d.loc[("DEMO_A", day("2024-01-15")), "close_repaired"]
assert np.isclose(d.loc[("DEMO_A", day("2024-01-16")), "close"], truth.loc[("DEMO_A", day("2024-01-16")), "close"])
assert d.loc[("DEMO_A", day("2024-01-16")), "low_repaired"]
assert np.isfinite(d.loc[("DEMO_A", day("2024-01-17")), "high"]) and d.loc[("DEMO_A", day("2024-01-17")), "high_repaired"]
assert np.isclose(d.loc[("DEMO_B", day("2024-01-19")), "adj_close"], truth.loc[("DEMO_B", day("2024-01-19")), "adj_close"])
assert d.loc[("DEMO_A", day("2024-02-06")), "close_repaired"]
assert abs(d.loc[("DEMO_A", day("2024-02-06")), "close"] / true_close - 1) < 0.03
assert (d["low"] <= d["open"]).all() and (d["open"] <= d["high"]).all()
assert (d["low"] <= d["close"]).all() and (d["close"] <= d["high"]).all()
print("All checks on the small example passed.")

All checks on the small example passed.


In [16]:
clean, excluded, counts = clean_data(raw, ref)
display(counts.to_frame("rows"))
display(excluded["reason"].value_counts())
display(excluded.loc[excluded["reason"] != "weekend date", ["ticker", "ts", "raw_ticker", "open", "high", "low",
                                                            "close", "reason"]])

,rows
raw rows,397711
removed: exact duplicates,29460
removed: all raw prices missing,0
price errors interpolated,4
"excluded (kept apart, with a reason)",389
kept in the cleaned table,367862


reason
weekend date                       384
no earlier price to repair from      5
Name: count, dtype: int64

,ticker,ts,raw_ticker,open,high,low,close,reason
384,CVX,1962-01-02,CVX,NaN,3.30000,3.24000,3.30000,no earlier price to repair from
385,JNJ,1962-01-02,JNJ,NaN,0.22338,0.22222,0.22338,no earlier price to repair from
386,MCD,1966-07-05,MCD,NaN,0.27366,0.26749,0.26955,no earlier price to repair from
387,MRK,1962-01-02,MRK,NaN,0.39703,0.39317,0.39537,no earlier price to repair from
388,PG,1962-01-02,PG,NaN,1.44000,1.42000,1.43000,no earlier price to repair from


In [17]:
flags = [c for c in clean.columns if c.endswith("_repaired") or c.endswith("_flag")]
display(clean[flags].sum().rename("rows flagged"))

open_repaired              4
high_repaired              4
low_repaired               4
close_repaired             4
adj_close_repaired     28767
volume_repaired            0
zero_volume_flag          72
extreme_return_flag     5182
Name: rows flagged, dtype: int64

The check with ohlc finds nothing: after removing the duplicates, ohlc_hw has the same prices as ohlc, so this check cannot prove that a move is real.

The volume check finds 4 errors, interpolated: TRV on 1976-10-28, 1978-12-20 and 1981-08-10 (the close doubles for one day) and AXP on 1972-12-14 (-13% for one day). For TRV the factor of ohlc is wrong too, so I interpolate the adjusted close instead of using the factor.

I keep the other 5,182 flags: many are known crashes (1987, 2008, 2020), and deleting them would hide risk. There are more flags than before (5,056) because the false TRV days made its std too large.

Not sure: MCD falls by 75% on 1968-01-02 in both tables, with volume. Probably a split never adjusted, but without proof I keep it with a flag.

In [18]:
largest = clean.loc[clean["extreme_return_flag"]].copy()
largest["return"] = flag_extreme_returns(clean)[0][clean["extreme_return_flag"]]
display(largest.reindex(largest["return"].abs().sort_values(ascending=False).index)[
    ["ticker", "ts", "close", "return"]].head(8))

,ticker,ts,close,return
213569,MCD,1968-01-02,0.174900,-0.753620
4999,AAPL,2000-09-29,0.459820,-0.518695
280553,NVDA,2000-03-07,0.173570,0.424106
281662,NVDA,2004-08-06,0.078584,-0.352314
336115,UNH,2008-10-13,23.380000,0.347550
22610,AMZN,2001-11-26,0.610500,0.344714
330824,UNH,1987-10-20,0.097656,-0.342118
22454,AMZN,2001-04-09,0.559000,0.335723


In [19]:
prices = clean[PRICES + ["adj_close"]]
checks = {
    "no missing price (open, high, low, close, adj_close)": not prices.isna().any().any(),
    "all prices finite": bool(np.isfinite(prices).all().all()),
    "all prices positive": bool((prices > 0).all().all()),
    "unique (ticker, ts)": not clean.duplicated(KEY).any(),
    "volume finite and >= 0": bool(np.isfinite(clean["volume"]).all() and (clean["volume"] >= 0).all()),
    "low <= open <= high": bool(((clean["low"] <= clean["open"]) & (clean["open"] <= clean["high"])).all()),
    "low <= close <= high": bool(((clean["low"] <= clean["close"]) & (clean["close"] <= clean["high"])).all()),
    "no weekend date": bool((clean["ts"].dt.dayofweek < 5).all()),
}
display(pd.Series(checks, name="passed").to_frame())
assert all(checks.values())

,passed
"no missing price (open, high, low, close, adj_close)",True
all prices finite,True
all prices positive,True
"unique (ticker, ts)",True
volume finite and >= 0,True
low <= open <= high,True
low <= close <= high,True
no weekend date,True


In [20]:
after = detect_issues(clean, ref)["issue_type"].value_counts().rename("after")
before = quality_report_before["issue_type"].value_counts().rename("before")
display(pd.concat([before, after], axis=1).fillna(0).astype(int))
print("Missing trading days after cleaning:", len(find_missing_days(clean, calendar)))

,before,after
issue_type,,
exact_duplicate,29460,0
missing_adj_close,18374,0
adj_close_inconsistent,10417,4
extreme_return,5056,5182
weekend_date,384,0
zero_volume,75,72
invalid_price,5,0
open_outside_range,5,0


Missing trading days after cleaning: 0


In [21]:
stats = pd.DataFrame({
    "rows": [len(raw), len(clean)],
    "tickers": [raw["ticker"].nunique(), clean["ticker"].nunique()],
    "missing values": [int(raw[VALUES].isna().sum().sum()), int(clean[VALUES].isna().sum().sum())],
    "max adj_close": [raw["adj_close"].max(), clean["adj_close"].max()],
    "min open": [raw["open"].min(), clean["open"].min()],
}, index=["before", "after"])
display(stats)

,rows,tickers,missing values,max adj_close,min open
before,397711,150,19855,10442.510783,0.000000
after,367862,30,0,605.400000,0.005208


The 4 adj_close_inconsistent left are the 4 interpolated errors, where the factor of ohlc is wrong.

Dates saved as text YYYY-MM-DD so the SQL filter of the report works, flags saved as 0 or 1.

In [22]:
to_save = clean[KEY + VALUES + flags + ["raw_ticker"]].copy()
to_save["ts"] = to_save["ts"].dt.strftime("%Y-%m-%d")
to_save["volume"] = to_save["volume"].round().astype("int64")
to_save[flags] = to_save[flags].astype(int)
to_save.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)

saved = pd.read_sql_query("SELECT COUNT(*) AS n, MIN(ts) AS first, MAX(ts) AS last FROM ohlc_hw_cleaned", conn)
raw_rows_now = pd.read_sql_query("SELECT COUNT(*) AS n FROM ohlc_hw", conn)["n"].iloc[0]
display(saved)
assert saved["n"].iloc[0] == len(clean) and raw_rows_now == len(raw)
print("Saved", len(clean), "rows in ohlc_hw_cleaned. ohlc_hw still has", raw_rows_now, "rows (not changed).")
conn.close()

,n,first,last
0,367862,1962-01-02,2024-12-06


Saved 367862 rows in ohlc_hw_cleaned. ohlc_hw still has 397711 rows (not changed).
